In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ==========================================
# PART 1: DATA PIPELINE (Robust Fix)
# ==========================================

def load_clean_split_data(file_paths, test_size=0.15):
    X_list = []
    y_list = []

    print("--- 1. Loading Data ---")
    for path in file_paths:
        try:
            data = np.load(path)
            if 'X' in data:
                X_curr, y_curr = data['X'], data['y']
            elif 'boards' in data:
                X_curr, y_curr = data['boards'], data['moves']
            else:
                X_curr, y_curr = data['arr_0'], data['arr_1']

            # Flatten to allow deduplication
            if X_curr.ndim > 2:
                X_curr = X_curr.reshape(X_curr.shape[0], -1)

            X_list.append(X_curr)
            y_list.append(y_curr)
            print(f"Loaded {path}: {len(X_curr)} samples")
        except Exception as e:
            print(f"Skipping {path}: {e}")

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)

    print("--- 2. Deduplicating (Majority Vote) ---")
    df = pd.DataFrame(X_all)
    df['move'] = y_all

    feature_cols = list(range(X_all.shape[1]))
    df_clean = df.groupby(feature_cols)['move'].agg(lambda x: x.mode()[0]).reset_index()

    X_clean = df_clean[feature_cols].values
    y_clean = df_clean['move'].values.astype(int)
    print(f"Cleaned size: {len(X_clean)}")

    print(f"--- 3. Creating Out-of-Sample Test Set ({test_size*100}%) ---")
    X_train_val, X_test_oos, y_train_val, y_test_oos = train_test_split(
        X_clean, y_clean, test_size=test_size, random_state=42, stratify=y_clean
    )

    return X_train_val, y_train_val, X_test_oos, y_test_oos

def process_and_augment(X_raw, y_raw, augment=True):
    processed_X = []
    processed_y = []

    # Detect Input Type: Raw (42) or Processed (84)
    input_size = X_raw.shape[1]
    is_raw = (input_size == 42)

    for i in range(len(X_raw)):
        # --- Handle Raw Data (42 integers: -1, 0, 1) ---
        if is_raw:
            board = X_raw[i].reshape(6, 7)
            # Perspective Fix
            if np.sum(board) != 0: board = board * -1
            # Encode
            my_pieces = (board == 1).astype(int)
            enemy_pieces = (board == -1).astype(int)
            formatted = np.stack([my_pieces, enemy_pieces], axis=-1)

        # --- Handle Processed Data (84 floats: 0.0, 1.0) ---
        else:
            # Already 6x7x2, just reshape
            formatted = X_raw[i].reshape(6, 7, 2)
            # Assumption: Ultimate dataset is already perspective-fixed

        # Add Original
        processed_X.append(formatted)
        processed_y.append(y_raw[i])

        # Add Augmentation (Flip)
        if augment:
            # Flip horizontal (axis 1)
            flipped_board = np.flip(formatted, axis=1)
            # Flip move (0->6, 1->5...)
            flipped_move = 6 - y_raw[i]

            processed_X.append(flipped_board)
            processed_y.append(flipped_move)

    return np.array(processed_X), np.array(processed_y)

# ==========================================
# PART 2: RESNET-20 MODEL (Fixed Filters)
# ==========================================

def resnet_layer(inputs, num_filters=16, kernel_size=3, strides=1, activation='relu', batch_normalization=True):
    conv = layers.Conv2D(num_filters, kernel_size=kernel_size, strides=strides, padding='same', kernel_initializer='he_normal',
                         kernel_regularizer=regularizers.l2(1e-3))  # INCREASED from 1e-4
    # ... rest stays the same
    x = inputs
    if conv: x = conv(x)
    if batch_normalization: x = layers.BatchNormalization()(x)
    if activation is not None: x = layers.Activation(activation)(x)
    return x

def build_resnet20(input_shape=(6, 7, 2)):
    inputs = layers.Input(shape=input_shape)

    num_filters = 32
    x = resnet_layer(inputs=inputs, num_filters=num_filters)

    depth = 20
    num_res_blocks = int((depth - 2) / 6)

    for stack in range(3):
        for res_block in range(num_res_blocks):
            strides = 1
            if stack > 0 and res_block == 0: strides = 1

            y = resnet_layer(inputs=x, num_filters=num_filters, strides=strides)
            y = resnet_layer(inputs=y, num_filters=num_filters, activation=None)

            if stack > 0 and res_block == 0:
                x = resnet_layer(inputs=x, num_filters=num_filters, kernel_size=1,
                                 strides=strides, activation=None, batch_normalization=False)

            x = layers.add([x, y])
            x = layers.Activation('relu')(x)

        # ADD: Spatial dropout after each stack
        x = layers.SpatialDropout2D(0.1)(x)

        num_filters *= 2

    x = layers.AveragePooling2D(pool_size=2)(x)
    x = layers.Flatten()(x)

    # CHANGE: Add a dense layer with dropout for better feature learning
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-3))(x)
    x = layers.Dropout(0.3)(x)  # Reduced from 0.5

    outputs = layers.Dense(7, activation='softmax', kernel_initializer='he_normal')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# ==========================================
# PART 3: EXECUTION
# ==========================================

# 1. LOAD & SPLIT
# This list can now handle BOTH 'ultimate' (84-size) and 'dedup' (42-size) files safely
files = ['connect4_dedup_574k.npz'] #'connect4_final_dedup.npz', 'connect4_dedup_574k', 'connect4_ultimate_dataset'
X_tv_raw, y_tv_raw, X_test_raw, y_test_raw = load_clean_split_data(files, test_size=0.15)

# 2. PROCESS & AUGMENT
print("Processing Training Data...")
X_train, y_train = process_and_augment(X_tv_raw, y_tv_raw, augment=True)

print("Processing Test Data...")
X_test, y_test = process_and_augment(X_test_raw, y_test_raw, augment=False)

print(f"Final Train Shape: {X_train.shape}")
print(f"Final Test Shape:  {X_test.shape}")

# 3. BUILD & TRAIN
model = build_resnet20()
model.compile(optimizer=tf.keras.optimizers.Adam(0.001, clipnorm=1.0), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),  # More patience
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)  # Lower floor
]

history = model.fit(
    X_train, y_train,
    epochs=60,
    batch_size=128,
    validation_split=0.2,
    callbacks=callbacks_list,
    verbose=1
)

# 4. FINAL EXAM
print("\n--- Running Final Out-of-Sample Test ---")
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"FINAL TEST ACCURACY: {test_acc*100:.2f}%")

model.save('resnet20_final_updated.h5')

--- 1. Loading Data ---
Loaded connect4_dedup_574k.npz: 574202 samples
--- 2. Deduplicating (Majority Vote) ---
Cleaned size: 574202
--- 3. Creating Out-of-Sample Test Set (15.0%) ---
Processing Training Data...
Processing Test Data...
Final Train Shape: (976142, 6, 7, 2)
Final Test Shape:  (86131, 6, 7, 2)
Epoch 1/60
6101/6101 ━━━━━━━━━━━━━━━━━━━━ 161s 22ms/step - accuracy: 0.4211 - loss: 2.2926 - val_accuracy: 0.5968 - val_loss: 1.2828 - learning_rate: 0.0010
Epoch 2/60
6101/6101 ━━━━━━━━━━━━━━━━━━━━ 110s 18ms/step - accuracy: 0.6081 - loss: 1.2607 - val_accuracy: 0.6242 - val_loss: 1.1609 - learning_rate: 0.0010
Epoch 3/60
6101/6101 ━━━━━━━━━━━━━━━━━━━━ 109s 18ms/step - accuracy: 0.6288 - loss: 1.1808 - val_accuracy: 0.6403 - val_loss: 1.1128 - learning_rate: 0.0010
Epoch 4/60
6101/6101 ━━━━━━━━━━━━━━━━━━━━ 108s 18ms/step - accuracy: 0.6367 - loss: 1.1477 - val_accuracy: 0.6403 - val_loss: 1.1122 - learning_rate: 0.0010
Epoch 5/60
6101/6101 ━━━━━━━━━━━━━━━━━━━━ 108s 18ms/step - accu

FINAL TEST ACCURACY: 70.07%
